In [13]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import pandas as pd

train_mgsd = pd.read_csv("data/mgsd.csv")
train_mentalmanip = pd.read_csv("data/mentalmanip.csv")
train_mmlu = pd.read_csv("data/samples_mmlu_full.csv")

examples_mgsd=pd.read_csv("data/mgsd_examples.csv")
examples_mentalmanip=pd.read_csv("data/mentalmanip_examples.csv")
examples_mmlu=pd.read_csv("data/samples_mmlu_full_examples.csv")

test_mgsd_load = pd.read_csv("data/mgsd_test.csv")
test_mentalmanip_load = pd.read_csv("data/mentalmanip_test.csv")
test_mmlu_load = pd.read_csv("data/samples_mmlu_full_test.csv")

test_mgsd = test_mgsd_load[:500]
test_mentalmanip = test_mentalmanip_load[:500]
test_mmlu = test_mmlu_load[:500]

In [3]:
from analysis_tools import (
    load_and_merge_profiles
    )
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from cases.mmlu_case import mmlu_case

from profiles.profile_sets import PERSON_ETHNICS

ALLOWED_MODELS = {"openai_4.1_mini"}

model_foldername = "openai_4.1_mini"

if model_foldername not in ALLOWED_MODELS:
    raise ValueError(f"Invalid model: {model_foldername}. Must be one of {ALLOWED_MODELS}")


merged_df_mgsd = load_and_merge_profiles(
    base_file_path= f"results/{model_foldername}/few_shot/classic/results_stereotype_few_shot_prompt_short_3examples_binary.csv",
    role_playing_glob_pattern=f"results/{model_foldername}/few_shot/role_playing_ethnics/*/results_stereotype_few_shot_prompt_short_3examples_binary.csv",
    sample_df=train_mgsd,
    case=stereotypes_case
)

merged_df_mentalmanip = load_and_merge_profiles(
    base_file_path= f"results/{model_foldername}/few_shot/classic/results_manipulation_few_shot_prompt_short_3examples.csv",
    role_playing_glob_pattern=f"results/{model_foldername}/few_shot/role_playing_ethnics/*/results_manipulation_few_shot_prompt_short_3examples_binary.csv",
    sample_df=train_mentalmanip,
    case=manipulation_case
)

merged_df_mmlu = load_and_merge_profiles(
    base_file_path=f"results/{model_foldername}/few_shot/classic/results_mmlu_few_shot_3examples_full.csv",
    role_playing_glob_pattern=f"results/{model_foldername}/few_shot/role_playing_ethnics/*/results_mmlu_few_shot_3examples_full.csv",
    sample_df=train_mmlu,
    case=mmlu_case,
)

=== Found 60 profile result files.
=== Merged DataFrame ready with columns:
 ['sample_id', 'true_label', 'base_pred', 'stereotype_type', 'profile1', 'profile2', 'profile3', 'profile4', 'profile5', 'profile6', 'profile7', 'profile8', 'profile9', 'profile10', 'profile11', 'profile12', 'profile13', 'profile14', 'profile15', 'profile16', 'profile17', 'profile18', 'profile19', 'profile20', 'profile21', 'profile22', 'profile23', 'profile24', 'profile25', 'profile26', 'profile27', 'profile28', 'profile29', 'profile30', 'profile31', 'profile32', 'profile33', 'profile34', 'profile35', 'profile36', 'profile37', 'profile38', 'profile39', 'profile40', 'profile41', 'profile42', 'profile43', 'profile44', 'profile45', 'profile46', 'profile47', 'profile48', 'profile49', 'profile50', 'profile51', 'profile52', 'profile53', 'profile54', 'profile55', 'profile56', 'profile57', 'profile58', 'profile59', 'profile60', 'prompt_tokens__base_pred', 'completion_tokens__base_pred', 'tokens_used__base_pred', 'max_t

In [ ]:
from dotenv import load_dotenv

import os
import openai
from openai import OpenAI
from anthropic import Anthropic
from mistralai import Mistral
import cohere
import google.generativeai as genai
from xai_sdk import Client as XAIClient


load_dotenv()


ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_DEEPSEEK": "DeepSeek",
    "API_KEY_GROK": "Grok",
    "API_KEY_ANTHROPIC": "Anthropic",
    "API_KEY_GEMINI": "Gemini",
    "API_KEY_MISTRAL": "Mistral",
    "API_KEY_COHERE": "Cohere",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        print(f"Warning: {name} - API key missing in .env file.")
        continue

API_KEY_OPENAI = os.getenv("API_KEY_OPENAI")
API_KEY_DEEPSEEK = os.getenv("API_KEY_DEEPSEEK")
API_KEY_ANTHROPIC = os.getenv("API_KEY_ANTHROPIC")
API_KEY_GEMINI = os.getenv("API_KEY_GEMINI")
API_KEY_MISTRAL = os.getenv("API_KEY_MISTRAL")
API_KEY_COHERE = os.getenv("API_KEY_COHERE")
API_KEY_GROK = os.getenv("API_KEY_GROK")


backend_to_run = [
    #"openai-4.1-mini",
    #"openai-4o-mini",
    #"mistral-small-2506",
    #"mistral-small-2503",
    "anthropic-sonnet",
    #"deepseek-v3-chat",
    #"gemini-2.5-flash",
]

backends = {
    "openai-4.1-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4.1-mini-2025-04-14",
        "fname":   "openai_4.1_mini"
    },
    "openai-4o-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4o-mini-2024-07-18",
        "fname":   "openai_4o_mini"
    },

    "deepseek-v3-chat": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_DEEPSEEK, base_url="https://api.deepseek.com"),
        "model":   "deepseek-chat",
        "fname":   "deepseek_v3"
    },

    "anthropic-sonnet": {
        "provider": "anthropic",
        "client":  Anthropic(api_key=API_KEY_ANTHROPIC),
        "model":   "claude-3-7-sonnet-latest",
        "fname":   "anthropic_3_7_sonnet"
    },

    "gemini-2.5-flash": {
        "provider": "gemini",
        "client":  (genai.configure(api_key=API_KEY_GEMINI) or genai.GenerativeModel("gemini-2.5-flash")),
        "model":   "gemini-2.5-flash",
        "fname":   "google_gemini_2_5_flash"
    },

    "mistral-small-2506": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2506",
        "fname":   "mistral_small_2506"
    },
    "mistral-small-2503": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2503",
        "fname":   "mistral_small_2503"
    },
}

### Cleaning MMLU

Since we will only test on the two normative categories 'moral_scenarios' and 'professional_law', we can remove the best on the control categories

In [5]:
merged_df_mmlu_normative = merged_df_mmlu.copy()
merged_df_mmlu_normative = merged_df_mmlu_normative[merged_df_mmlu_normative['subject'].isin(['moral_scenarios', 'professional_law'])]

## Few-shot on test sets for offline tests

To compare the results obtained with our method and ascertain if they could lead to good results, we must first run our dataset on the test set and compare the offline results on our best 3 personalities for each dataset to the offline results of clustering smart routing.

In [6]:
def profile_metrics(df, profile_col):
    true = df["true_label"].astype(str)
    base = df["base_pred"].astype(str)
    prof = df[profile_col].astype(str)

    acc = (prof==true).mean()
    basea = (base==true).mean()
    rescue = ((base!=true)&(prof==true)).mean()
    extra = ((base==true)&(prof!=true)).mean()
    return {"acc":acc, "accDiff":acc-basea, "rescue":rescue, "extra":extra, "score":rescue-extra}

prof_cols = [c for c in merged_df_mgsd.columns if c.startswith("profile")]
rows = []
for c in prof_cols:
    m = profile_metrics(merged_df_mgsd, c)
    rows.append({"profile": c, **m})

prof_table = pd.DataFrame(rows).sort_values(["accDiff","acc"], ascending=False)
best_3_mgsd = prof_table.head(3)["profile"].tolist()
print("Best 3 profiles:", best_3_mgsd)

Best 3 profiles: ['profile43', 'profile51', 'profile1']


In [7]:
prof_cols = [c for c in merged_df_mentalmanip.columns if c.startswith("profile")]
rows = []
for c in prof_cols:
    m = profile_metrics(merged_df_mentalmanip, c)
    rows.append({"profile": c, **m})

prof_table = pd.DataFrame(rows).sort_values(["accDiff","acc"], ascending=False)
best_3_mentalmanip = prof_table.head(3)["profile"].tolist()
print("Best 3 profiles:", best_3_mentalmanip)

Best 3 profiles: ['profile4', 'profile6', 'profile45']


In [8]:
prof_cols = [c for c in merged_df_mmlu_normative.columns if c.startswith("profile")]
rows = []
for c in prof_cols:
    m = profile_metrics(merged_df_mmlu_normative, c)
    rows.append({"profile": c, **m})

prof_table = pd.DataFrame(rows).sort_values(["accDiff","acc"], ascending=False)
best_3_mmlu = prof_table.head(3)["profile"].tolist()
print("Best 3 profiles:", best_3_mmlu)

Best 3 profiles: ['profile3', 'profile52', 'profile51']


### Best profiles few shots on each test dataset

In [14]:
import pandas as pd
from tqdm import tqdm
import os

from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short

from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from cases.mmlu_case import mmlu_case
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from openai import RateLimitError
from data_loader import get_additional_fields
from profiles.profile_sets import PERSON_ETHNICS, PERSON_SYSTEMATIC, PERSON_SEEDS_CORE


prompt_type = "short"
max_tokens = 300
case_name_set = ["stereotype", "manipulation"] # "mmlu"

person_set = PERSON_ETHNICS


person_seeds = person_set.seeds

if person_seeds == PERSON_SEEDS_CORE:
    folder_role_playing = "role_playing_core" 
elif (person_seeds == PERSON_SYSTEMATIC.seeds):
    folder_role_playing = "role_playing_system"
elif (person_seeds == PERSON_ETHNICS.seeds):
    folder_role_playing = "role_playing_ethnics"
else:
    folder_role_playing = "role_playing"


for case_name in case_name_set:

    if case_name == "manipulation":
        case = manipulation_case
        task_definition = manipulation_definition_short
        data = test_mentalmanip
        few_shot_examples = examples_mentalmanip
        selected_profiles = best_3_mentalmanip

    elif case_name == "stereotype":
        case = stereotypes_case
        task_definition = stereotype_definition_short_binary
        data = test_mgsd
        few_shot_examples = examples_mgsd
        selected_profiles = best_3_mgsd

    elif case_name == "mmlu":
        case= mmlu_case
        task_definition = "" 
        data = test_mmlu
        few_shot_examples = examples_mmlu
        selected_profiles = best_3_mmlu      
    else:
        raise ValueError(f"Unknown case name: {case_name}")


    for person_key in selected_profiles:

        for role_playing in ["passive"]:

            type_suffix = "" if case_name =="mmlu" else "binary_"
            file_suffix = (
                f"results_{case_name}_few_shot_prompt_short_3examples_{type_suffix}test.csv"
            )
            output_file = f"results/{model_foldername}/clustering/{folder_role_playing}/{person_key}_{role_playing}/{file_suffix}"
            os.makedirs(os.path.dirname(output_file), exist_ok=True)

            classifier = FewShot(
                    case=case,
                    client=client,
                    model=model,
                    max_tokens=max_tokens,
                    task_definition=task_definition,
                    n_shots=3,
                    examples_df=few_shot_examples,
                    person_key=person_key,
                    role_playing=role_playing,
                    person_set=person_set
            )

            rows = []
            for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"{case_name} | {person_key} | few_shot | {role_playing}"):
                try:
                    text = row[case.input_col]                        
                    true_label = row[case.label_col]

                    if case_name == "mmlu":
                        pred, stats = classifier.classify(text, row=row.to_dict())
                        raw = pred.strip().upper()
                    else:
                        pred, stats = classifier.classify(text)
                        raw = pred.strip().lower()
                        raw = raw.capitalize()

                    mapped = case.label_map.get(raw)

                    if mapped is None:
                        print(f"[warn] Unmapped prediction '{raw}' (case={case_name}); skipping sample {idx}.")

                    additional = get_additional_fields(row, case_name)

                    rows.append({
                        "sample_id": idx,
                        "text": text,
                        "true_label": true_label,
                        "pred_label": mapped,
                        "max_tokens": classifier.max_tokens,
                        "tokens_used": stats["tokens_used"],
                        "prompt_tokens": stats["prompt_tokens"],
                        "completion_tokens": stats["completion_tokens"],
                        "latency": stats["latency"],
                        **additional,
                    })
                except RateLimitError as e:
                    print(f"Error : {e}")
                    print(f"WARNING: Rate limit hit at sample {idx}. Skipping.")
                    continue
                except Exception as e:
                    print(f"ERROR at sample {idx}: {e}")
                    continue

            df_out = pd.DataFrame(rows)
            df_out.to_csv(output_file, index=False)
            print(f"✅ Saved {len(df_out)} rows to {output_file}")

            if case_name in {"manipulation", "mmlu"}:
                y_true = df_out["true_label"].astype(int)
                y_pred = df_out["pred_label"].astype(int)
            else:
                y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()

            print("\n=== Classification Report ===")
            print(classification_report(y_true, y_pred))

            print("\n=== Confusion Matrix ===")
            labels = sorted(set(y_true) | set(y_pred))
            conf_matrix = confusion_matrix(y_true, y_pred)
            print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

            accuracy = accuracy_score(y_true, y_pred)
            print(f"\n=== Accuracy for {case_name} | {person_key} | few_shot | {role_playing}: {accuracy:.3f}")


stereotype | profile43 | few_shot | passive: 100%|██████████| 500/500 [06:09<00:00,  1.35it/s]


✅ Saved 500 rows to results/openai_4.1_mini/clustering/role_playing_ethnics/profile43_passive/results_stereotype_few_shot_prompt_short_3examples_binary_test.csv

=== Classification Report ===
              precision    recall  f1-score   support

  stereotype       0.72      0.65      0.69       260
   unrelated       0.66      0.73      0.69       240

    accuracy                           0.69       500
   macro avg       0.69      0.69      0.69       500
weighted avg       0.69      0.69      0.69       500


=== Confusion Matrix ===
            stereotype  unrelated
stereotype         170         90
unrelated           65        175

=== Accuracy for stereotype | profile43 | few_shot | passive: 0.690


stereotype | profile51 | few_shot | passive: 100%|██████████| 500/500 [05:22<00:00,  1.55it/s]


✅ Saved 500 rows to results/openai_4.1_mini/clustering/role_playing_ethnics/profile51_passive/results_stereotype_few_shot_prompt_short_3examples_binary_test.csv

=== Classification Report ===
              precision    recall  f1-score   support

  stereotype       0.72      0.67      0.69       260
   unrelated       0.67      0.71      0.69       240

    accuracy                           0.69       500
   macro avg       0.69      0.69      0.69       500
weighted avg       0.69      0.69      0.69       500


=== Confusion Matrix ===
            stereotype  unrelated
stereotype         175         85
unrelated           69        171

=== Accuracy for stereotype | profile51 | few_shot | passive: 0.692


stereotype | profile1 | few_shot | passive: 100%|██████████| 500/500 [08:11<00:00,  1.02it/s]


...

✅ Saved 500 rows to results/openai_4.1_mini/clustering/role_playing_ethnics/profile4_passive/results_manipulation_few_shot_prompt_short_3examples_binary_test.csv

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.64      0.61      0.63       242
           1       0.65      0.68      0.66       258

    accuracy                           0.65       500
   macro avg       0.65      0.64      0.64       500
weighted avg       0.65      0.65      0.65       500


=== Confusion Matrix ===
     0    1
0  148   94
1   83  175

=== Accuracy for manipulation | profile4 | few_shot | passive: 0.646


manipulation | profile6 | few_shot | passive: 100%|██████████| 500/500 [06:44<00:00,  1.24it/s]


✅ Saved 500 rows to results/openai_4.1_mini/clustering/role_playing_ethnics/profile6_passive/results_manipulation_few_shot_prompt_short_3examples_binary_test.csv

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.64      0.62      0.63       242
           1       0.66      0.67      0.67       258

    accuracy                           0.65       500
   macro avg       0.65      0.65      0.65       500
weighted avg       0.65      0.65      0.65       500


=== Confusion Matrix ===
     0    1
0  151   91
1   84  174

=== Accuracy for manipulation | profile6 | few_shot | passive: 0.650


manipulation | profile45 | few_shot | passive: 100%|██████████| 500/500 [05:50<00:00,  1.43it/s]

✅ Saved 500 rows to results/openai_4.1_mini/clustering/role_playing_ethnics/profile45_passive/results_manipulation_few_shot_prompt_short_3examples_binary_test.csv

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.64      0.59      0.61       242
           1       0.64      0.69      0.66       258

    accuracy                           0.64       500
   macro avg       0.64      0.64      0.64       500
weighted avg       0.64      0.64      0.64       500


=== Confusion Matrix ===
     0    1
0  142  100
1   81  177

=== Accuracy for manipulation | profile45 | few_shot | passive: 0.638


### Neutral profile few-shot on test dataset

In [15]:
import pandas as pd
from tqdm import tqdm
import os

from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short

from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from cases.mmlu_case import mmlu_case

from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from openai import RateLimitError
from data_loader import get_additional_fields
from profiles.profile_sets import PERSON_ETHNICS, PERSON_SYSTEMATIC, PERSON_SEEDS_CORE


prompt_type = "short"
max_tokens = 300
case_name_set = ["stereotype", "manipulation", "mmlu"]

person_set = PERSON_ETHNICS


person_seeds = person_set.seeds

if person_seeds == PERSON_SEEDS_CORE:
    folder_role_playing = "role_playing_core" 
elif (person_seeds == PERSON_SYSTEMATIC.seeds):
    folder_role_playing = "role_playing_system"
elif (person_seeds == PERSON_ETHNICS.seeds):
    folder_role_playing = "role_playing_ethnics"
else:
    folder_role_playing = "role_playing"


for case_name in case_name_set:

    if case_name == "manipulation":
        case = manipulation_case
        task_definition = manipulation_definition_short
        data = test_mentalmanip
        few_shot_examples = examples_mentalmanip
        selected_profiles = best_3_mentalmanip

    elif case_name == "stereotype":
        case = stereotypes_case
        task_definition = stereotype_definition_short_binary
        data = test_mgsd
        few_shot_examples = examples_mgsd
        selected_profiles = best_3_mgsd

    elif case_name == "mmlu":
        case= mmlu_case
        task_definition = "" 
        data = test_mmlu
        few_shot_examples = examples_mmlu
        selected_profiles = best_3_mmlu      
    else:
        raise ValueError(f"Unknown case name: {case_name}")




    for role_playing in ["none"]:
        person_key = None
        type_suffix = "" if case_name =="mmlu" else "binary_"
        file_suffix = (
            f"results_{case_name}_few_shot_prompt_short_3examples_{type_suffix}test.csv"
        )
        output_file = f"results/{model_foldername}/clustering/classic/{file_suffix}"
        os.makedirs(os.path.dirname(output_file), exist_ok=True)

        classifier = FewShot(
                case=case,
                client=client,
                model=model,
                max_tokens=max_tokens,
                task_definition=task_definition,
                n_shots=3,
                examples_df=few_shot_examples,
                person_key=person_key,
                role_playing=role_playing,
                person_set=person_set
            )

        rows = []
        for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"{case_name} | {person_key} | few_shot | {role_playing}"):
            try:
                text = row[case.input_col]                        
                true_label = row[case.label_col]

                if case_name == "mmlu":
                    pred, stats = classifier.classify(text, row=row.to_dict())
                    raw = pred.strip().upper()
                else:
                    pred, stats = classifier.classify(text)
                    raw = pred.strip().lower()
                    raw = raw.capitalize()

                mapped = case.label_map.get(raw)
                if mapped is None:
                    print(f"[warn] Unmapped prediction '{raw}' (case={case_name}); skipping sample {idx}.")

                additional = get_additional_fields(row, case_name)

                rows.append({
                    "sample_id": idx,
                    "text": text,
                    "true_label": true_label,
                    "pred_label": mapped,
                    "max_tokens": classifier.max_tokens,
                    "tokens_used": stats["tokens_used"],
                    "prompt_tokens": stats["prompt_tokens"],
                    "completion_tokens": stats["completion_tokens"],
                    "latency": stats["latency"],
                    **additional,
                })
            except RateLimitError as e:
                print(f"Error : {e}")
                print(f"WARNING: Rate limit hit at sample {idx}. Skipping.")
                continue
            except Exception as e:
                print(f"ERROR at sample {idx}: {e}")
                continue

        df_out = pd.DataFrame(rows)
        df_out.to_csv(output_file, index=False)
        print(f"✅ Saved {len(df_out)} rows to {output_file}")

        if case_name in {"manipulation", "mmlu"}:
            y_true = df_out["true_label"].astype(int)
            y_pred = df_out["pred_label"].astype(int)
        else:
            y_true = df_out["true_label"].astype(str).str.strip().str.lower()
            y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()

        print("\n=== Classification Report ===")
        print(classification_report(y_true, y_pred))

        print("\n=== Confusion Matrix ===")
        labels = sorted(set(y_true) | set(y_pred))
        conf_matrix = confusion_matrix(y_true, y_pred)
        print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

        accuracy = accuracy_score(y_true, y_pred)
        print(f"\n=== Accuracy for {case_name} | {person_key} | few_shot | {role_playing}: {accuracy:.3f}")

stereotype | None | few_shot | none: 100%|██████████| 500/500 [06:29<00:00,  1.28it/s]


✅ Saved 500 rows to results/openai_4.1_mini/clustering/classic/results_stereotype_few_shot_prompt_short_3examples_binary_test.csv

=== Classification Report ===
              precision    recall  f1-score   support

  stereotype       0.74      0.57      0.65       260
   unrelated       0.63      0.78      0.70       240

    accuracy                           0.67       500
   macro avg       0.69      0.68      0.67       500
weighted avg       0.69      0.67      0.67       500


=== Confusion Matrix ===
            stereotype  unrelated
stereotype         149        111
unrelated           52        188

=== Accuracy for stereotype | None | few_shot | none: 0.674


manipulation | None | few_shot | none: 100%|██████████| 500/500 [05:49<00:00,  1.43it/s]


✅ Saved 500 rows to results/openai_4.1_mini/clustering/classic/results_manipulation_few_shot_prompt_short_3examples_binary_test.csv

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.63      0.66      0.65       242
           1       0.67      0.64      0.65       258

    accuracy                           0.65       500
   macro avg       0.65      0.65      0.65       500
weighted avg       0.65      0.65      0.65       500


=== Confusion Matrix ===
     0    1
0  159   83
1   92  166

=== Accuracy for manipulation | None | few_shot | none: 0.650


mmlu | None | few_shot | none: 100%|██████████| 500/500 [07:28<00:00,  1.11it/s]

✅ Saved 500 rows to results/openai_4.1_mini/clustering/classic/results_mmlu_few_shot_prompt_short_3examples_test.csv

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.60      0.65      0.63       133
           1       0.67      0.64      0.66       123
           2       0.68      0.63      0.66       126
           3       0.59      0.60      0.60       118

    accuracy                           0.63       500
   macro avg       0.64      0.63      0.63       500
weighted avg       0.64      0.63      0.63       500


=== Confusion Matrix ===
    0   1   2   3
0  87  15  13  18
1  16  79  13  15
2  19  11  80  16
3  22  13  12  71

=== Accuracy for mmlu | None | few_shot | none: 0.634


# Clustering/Smart Routing

In [19]:
from profiles.profile_sets import PERSON_ETHNICS
from clustering.few_shot_integration import *
from clustering.clustering_train_test import run_train_test_pipeline
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
import os, json
from pathlib import Path
import pandas as pd

person_set = PERSON_ETHNICS

CASES_TO_RUN = ["stereotype"] # "stereotype", "manipulation",
EVAL_MODE = "policy_online"
WITH_FDR = False
RISK_CAP = None               
CATEGORY_RISK_CAP = None      
MIN_CATEGORY_N = 20

SAVE_DIR = "results/openai_4.1_mini/clustering/clustering_models"
PERF_CSV = None

perf_df = None
if PERF_CSV:
    import pandas as pd
    perf_df = pd.read_csv(PERF_CSV)

cases = {}
try:
    cases["stereotype"] = {
        "case": stereotypes_case,
            "train": train_mgsd,
            "train_merged": merged_df_mgsd,     
            "test":  test_mgsd,
            "examples": examples_mgsd,
            "task_def": stereotype_definition_short_binary,
    }
except NameError as e:
    print("Skipping stereotype:", e)
    pass
try:
    cases["manipulation"] = {
            "case": manipulation_case,
            "train": train_mentalmanip,
            "train_merged": merged_df_mentalmanip, 
            "test":  test_mentalmanip,
            "examples": examples_mentalmanip,
            "task_def": manipulation_definition_short,
        }
except NameError as e:
        print("Skipping manipulation:", e)
        pass
try:
    cases["mmlu"] = {
            "case": mmlu_case,
            "train": train_mmlu,
            "train_merged": merged_df_mmlu_normative,
            "test":  test_mmlu,
            "examples": examples_mmlu,
            "task_def": "", 
    }
except NameError as e:
        print("Skipping stereotype:", e)
        pass


selected = [c for c in CASES_TO_RUN if c in cases]
print(f"Selected cases: {selected}")
if not selected:
    raise RuntimeError("No matching cases available. Ensure train/test DFs & case configs are loaded.")

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
all_results = {}

for EVAL_MODE in ["policy_online"]:
    for WITH_FDR in [False, True]:
        for name in selected:
            print("\n" + "="*60)
            print(f"Running case : {name} | mode={EVAL_MODE} | FDR={WITH_FDR}")
            cfg = cases[name]
            case = cfg["case"]
            train_df = cfg["train_merged"]
            train_df_text = cfg["train"] 
            test_df = cfg["test"]
            examples_df = cfg["examples"]
            task_def = cfg["task_def"]

            # Policy used during TRAIN to build deployment plans
            policy = RoutingPolicyConfig(
                enable_pareto=False,
                lambda_tok=5e-4,
                lambda_extra=2.0,
                risk_cap=RISK_CAP,
                category_risk_cap=CATEGORY_RISK_CAP,
                min_category_n=MIN_CATEGORY_N,
                require_fdr=WITH_FDR,
                q_threshold=0.10,
                enable_tier3_exploratory=False,
            )

            out_dir = os.path.join(SAVE_DIR, f"{EVAL_MODE}", f"fdr_{str(WITH_FDR).lower()}", name)
            Path(out_dir).mkdir(parents=True, exist_ok=True) 

            gen_fn = make_online_gen_fn(
                case=case,
                client=client,
                model=model,
                person_set=person_set,
                examples_df=examples_df,
                task_definition=task_def,
                n_shots=3,
                max_tokens=300,
            )

            # Train + baseline test (pipeline object returned)
            pipe_out = run_train_test_pipeline(
                train_df=cases[name]["train_merged"],  
                test_df=cases[name]["test"],
                sample_df=cases[name]["train"],      
                case=case,
                person_set=person_set,
                save_dir=out_dir,
                perf_df=perf_df,
                routing_policy=policy,      # <-- pass the policy in
                eval_mode="policy_online",
                gen_fn=gen_fn,
            )
            pipeline = pipe_out["pipeline"]

            # Online generator (only if needed)
            gen_fn = None
            if EVAL_MODE == "policy_online":
                gen_fn = make_online_gen_fn(
                    case=case,
                    client=client,
                    model=model,
                    person_set=person_set,
                    examples_df=examples_df,
                    task_definition=task_def,
                    n_shots=3,
                    max_tokens=300,
                )

            # Final evaluation in desired mode
            from datetime import datetime

            RUN_TAG = f"{name}_{EVAL_MODE}_fdr_{str(WITH_FDR).lower()}_{datetime.now():%Y%m%d-%H%M%S}"

            test_out = pipeline.test(
                test_df=test_df,
                sample_df=train_df,
                case=case,
                person_set=person_set,
                eval_mode=EVAL_MODE,
                gen_fn=gen_fn,
                label_map=None,
                save_internal=False,     
                run_tag=RUN_TAG,    
            )

            # Ensure per-case dir exists
            Path(out_dir).mkdir(parents=True, exist_ok=True)

            # Save policy decisions (offline or online) + source plan
            policy_csv = os.path.join(out_dir, f"{RUN_TAG}.csv")
            test_out["policy_df"].to_csv(policy_csv, index=False)
            print(f"[✓] Saved policy decisions → {policy_csv}")

            tok = test_out.get("policy_token_logs")
            if tok:
                tok_csv = os.path.join(out_dir, f"{RUN_TAG}_token_logs.csv")
                pd.DataFrame(tok).to_csv(tok_csv, index=False)
                print(f"[✓] Saved policy token logs → {tok_csv}")

            summary = {
                "train_metrics": pipe_out["train"]["training_metrics"],
                "test_metrics": test_out["test_metrics"],
                "policy_metrics": test_out.get("policy_metrics"),
                "cluster_distribution": test_out["cluster_distribution"],
            }
            all_results[f"{name}__mode={EVAL_MODE}__fdr={WITH_FDR}"] = summary

            out_path = os.path.join(out_dir, f"summary_{name}.json")
            with open(out_path, "w") as f:
                json.dump(summary, f, indent=2)
            print(f"[✓] Saved summary → {out_path}")

            pm = summary.get("policy_metrics") or {}
            te = summary["test_metrics"]
            line = f"- {name}: train={summary['train_metrics'].get('accuracy', float('nan')):.3f} | test={te.get('accuracy', float('nan')):.3f}"
            if pm:
                line += f" | online_acc={pm.get('accuracy', float('nan')):.3f} | Δacc={pm.get('accuracy_improvement', float('nan')):+.3f}"
            print(line)


Selected cases: ['stereotype']

Running case : stereotype | mode=policy_online | FDR=False

TRAINING ADAPTIVE CLUSTERING MODEL - model_v1

Generating embeddings for 500 samples...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]


Finding optimal clustering...


Grid search: k:   0%|          | 0/10 [00:00<?, ?it/s]


Tier-2 Ensemble Analysis for 3 clusters:

  Analyzing Cluster 0 (n=96):
Building trait groups with keys: ('gender', 'ethnicity', 'age')
Found 60 profile columns
Discovered trait values:
  gender: ['man', 'woman']
  ethnicity: ['asian', 'black', 'indian', 'latine', 'middle_eastern', 'white']
  age: ['20', '25', '35', '45', '55']
Created 60 trait groups:
  man_asian_20: 1 profiles
  man_asian_25: 1 profiles
  man_asian_35: 1 profiles
  man_asian_45: 1 profiles
  man_asian_55: 1 profiles
  man_black_20: 1 profiles
  man_black_25: 1 profiles
  man_black_35: 1 profiles
  man_black_45: 1 profiles
  man_black_55: 1 profiles
  man_indian_20: 1 profiles
  man_indian_25: 1 profiles
  man_indian_35: 1 profiles
  man_indian_45: 1 profiles
  man_indian_55: 1 profiles
  man_latine_20: 1 profiles
  man_latine_25: 1 profiles
  man_latine_35: 1 profiles
  man_latine_45: 1 profiles
  man_latine_55: 1 profiles
  man_middle_eastern_20: 1 profiles
  man_middle_eastern_25: 1 profiles
  man_middle_eastern_3

...

[saved] results/openai_4.1_mini/clustering/clustering_models/policy_online/fdr_true/stereotype/stereotype_8_fdr_true_20250910-175707_policy_online_decisions.csv
[saved] results/openai_4.1_mini/clustering/clustering_models/policy_online/fdr_true/stereotype/stereotype_8_fdr_true_20250910-175707_policy_online_token_logs.csv

Policy deployment (online) performance:
  Accuracy: 0.696
  Baseline: 0.672
  ΔAcc: +0.024
  Rescue: 0.058 | Extra: 0.034

Testing on new data - mode: policy_online

Embedding 500 test samples...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Assigning to clusters...

Cluster distribution in test set:
  Cluster 0: 84 samples (16.8%)
  Cluster 1: 87 samples (17.4%)
  Cluster 2: 78 samples (15.6%)
  Cluster 3: 24 samples (4.8%)
  Cluster 4: 49 samples (9.8%)
  Cluster 5: 68 samples (13.6%)
  Cluster 6: 30 samples (6.0%)
  Cluster 7: 80 samples (16.0%)



Average distance to centroids: 0.914



Policy deployment (online) performance:
  Accuracy: 0.694
  Baseline: 0.668
  ΔAcc: +0.026
  Rescue: 0.056 | Extra: 0.030
[✓] Saved policy decisions → results/openai_4.1_mini/clustering/clustering_models/policy_online/fdr_true/stereotype/stereotype_policy_online_fdr_true_20250910-175707.csv
[✓] Saved policy token logs → results/openai_4.1_mini/clustering/clustering_models/policy_online/fdr_true/stereotype/stereotype_policy_online_fdr_true_20250910-175707_token_logs.csv
[✓] Saved summary → results/openai_4.1_mini/clustering/clustering_models/policy_online/fdr_true/stereotype/summary_stereotype.json
- stereotype: train=0.762 | test=0.694 | online_acc=0.694 | Δacc=+0.026
